# 第 7 课：Agentic RAG

预计用时：90–120 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 理解普通 RAG 与 Agentic RAG 的边界
- 实现最小内存向量库
- 让 Agent 自主决定何时及如何检索

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 1. 先理解概念

普通 RAG 通常固定检索一次再生成；Agentic RAG 可以判断是否检索、改写查询并多轮补证据。自主性越高，越需要限制检索次数、去重证据并要求回答引用来源。

### 本课路线

1. 定义带来源的文档模型
2. 批量生成并归一化向量
3. 实现 Top-K 相似度搜索
4. 把检索器注册成工具
5. 添加文档并运行带引用问答


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 准备代码


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 准备代码


In [ ]:
class AgentLimits(BaseModel):
    model_config = ConfigDict(extra='forbid')
    max_steps: int = Field(default=8, ge=1, le=30)
    model_timeout_s: float = Field(default=45, gt=0, le=300)
    tool_timeout_s: float = Field(default=15, gt=0, le=120)
    total_timeout_s: float = Field(default=120, gt=0, le=600)

class ToolResult(BaseModel):
    ok: bool
    data: Any = None
    error: str | None = None
    retryable: bool = False

ToolHandler = Callable[[BaseModel], Awaitable[Any]]

@dataclass
class RegisteredTool:
    name: str
    description: str
    args_model: type[BaseModel]
    handler: ToolHandler
    side_effect: bool = False

    def openai_schema(self) -> dict[str, Any]:
        schema = self.args_model.model_json_schema()
        schema['additionalProperties'] = False
        return {
            'type': 'function',
            'function': {
                'name': self.name,
                'description': self.description,
                'parameters': schema,
            },
        }

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, RegisteredTool] = {}

    def register(self, tool: RegisteredTool) -> None:
        if tool.name in self._tools:
            raise ValueError(f'工具重复注册: {tool.name}')
        self._tools[tool.name] = tool

    @property
    def schemas(self) -> list[dict[str, Any]]:
        return [tool.openai_schema() for tool in self._tools.values()]

    async def execute(self, name: str, raw_arguments: str, timeout_s: float) -> ToolResult:
        tool = self._tools.get(name)
        if tool is None:
            return ToolResult(ok=False, error=f'未知工具: {name}', retryable=False)
        try:
            arguments = json.loads(raw_arguments or '{}')
            validated = tool.args_model.model_validate(arguments)
        except json.JSONDecodeError as exc:
            return ToolResult(ok=False, error=f'工具参数不是合法 JSON: {exc}')
        except ValidationError as exc:
            return ToolResult(ok=False, error=f'工具参数校验失败: {exc}')
        try:
            async with asyncio.timeout(timeout_s):
                value = await tool.handler(validated)
            return ToolResult(ok=True, data=value)
        except TimeoutError:
            return ToolResult(ok=False, error=f'工具 {name} 执行超时', retryable=True)
        except httpx.HTTPStatusError as exc:
            retryable = exc.response.status_code in {408, 429, 500, 502, 503, 504}
            return ToolResult(ok=False, error=f'上游 HTTP {exc.response.status_code}', retryable=retryable)
        except Exception as exc:
            return ToolResult(ok=False, error=f'{type(exc).__name__}: {exc}', retryable=False)


### 准备代码


In [ ]:
class MinimalAgent:
    def __init__(self, registry: ToolRegistry, limits: AgentLimits | None = None) -> None:
        self.registry = registry
        self.limits = limits or AgentLimits()

    async def run(self, user_input: str) -> str:
        require_api_key()
        messages: list[dict[str, Any]] = [
            {'role': 'system', 'content': '你是可靠的中文助手。需要外部事实或操作时调用工具；工具失败时不得编造结果。'},
            {'role': 'user', 'content': user_input},
        ]
        repeated_calls: dict[str, int] = {}

        async with asyncio.timeout(self.limits.total_timeout_s):
            for step in range(1, self.limits.max_steps + 1):
                async with asyncio.timeout(self.limits.model_timeout_s):
                    response = await client.chat.completions.create(
                        model=BAILIAN_MODEL,
                        messages=messages,
                        tools=self.registry.schemas or None,
                        tool_choice='auto' if self.registry.schemas else None,
                        temperature=0.2,
                    )
                message = response.choices[0].message
                assistant_message: dict[str, Any] = {
                    'role': 'assistant',
                    'content': message.content or '',
                }
                if message.tool_calls:
                    assistant_message['tool_calls'] = [tc.model_dump() for tc in message.tool_calls]
                messages.append(assistant_message)

                if not message.tool_calls:
                    return message.content or ''

                for call in message.tool_calls:
                    fingerprint = f'{call.function.name}:{call.function.arguments}'
                    repeated_calls[fingerprint] = repeated_calls.get(fingerprint, 0) + 1
                    if repeated_calls[fingerprint] > 2:
                        result = ToolResult(ok=False, error='相同工具调用重复过多，已阻止循环')
                    else:
                        result = await self.registry.execute(
                            call.function.name,
                            call.function.arguments,
                            self.limits.tool_timeout_s,
                        )
                    messages.append({
                        'role': 'tool',
                        'tool_call_id': call.id,
                        'content': result.model_dump_json(),
                    })

        raise RuntimeError(f'Agent 超过最大步骤数 {self.limits.max_steps}')


### 准备代码


In [ ]:
MEMORY_DB = WORKSPACE / 'memory.sqlite3'

class LongTermMemory:
    def __init__(self, path: Path = MEMORY_DB) -> None:
        self.path = path
        with sqlite3.connect(self.path) as conn:
            conn.executescript('''
                CREATE TABLE IF NOT EXISTS preferences (
                    user_id TEXT, key TEXT, value TEXT, confidence REAL, source TEXT, updated_at TEXT,
                    PRIMARY KEY(user_id, key)
                );
                CREATE TABLE IF NOT EXISTS memories (
                    id INTEGER PRIMARY KEY, user_id TEXT, text TEXT, embedding BLOB, created_at TEXT
                );
            ''')

    def upsert_preference(self, user_id: str, key: str, value: str, source: str, confidence: float = 1.0) -> None:
        if not 0 <= confidence <= 1:
            raise ValueError('confidence 必须在 0 到 1 之间')
        with sqlite3.connect(self.path) as conn:
            conn.execute('''
                INSERT INTO preferences VALUES (?, ?, ?, ?, ?, ?)
                ON CONFLICT(user_id, key) DO UPDATE SET
                  value=excluded.value, confidence=excluded.confidence,
                  source=excluded.source, updated_at=excluded.updated_at
            ''', (user_id, key, value, confidence, source, datetime.now(timezone.utc).isoformat()))

    def preferences(self, user_id: str) -> dict[str, str]:
        with sqlite3.connect(self.path) as conn:
            rows = conn.execute('SELECT key, value FROM preferences WHERE user_id=?', (user_id,)).fetchall()
        return dict(rows)

    async def embed(self, texts: list[str]) -> list[list[float]]:
        require_api_key()
        response = await client.embeddings.create(model=BAILIAN_EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    async def remember(self, user_id: str, text: str) -> int:
        vector = np.asarray((await self.embed([text]))[0], dtype=np.float32)
        with sqlite3.connect(self.path) as conn:
            cursor = conn.execute(
                'INSERT INTO memories(user_id, text, embedding, created_at) VALUES (?, ?, ?, ?)',
                (user_id, text, vector.tobytes(), datetime.now(timezone.utc).isoformat()),
            )
            return int(cursor.lastrowid)

    async def recall(self, user_id: str, query: str, k: int = 5) -> list[dict[str, Any]]:
        query_vec = np.asarray((await self.embed([query]))[0], dtype=np.float32)
        with sqlite3.connect(self.path) as conn:
            rows = conn.execute('SELECT id, text, embedding FROM memories WHERE user_id=?', (user_id,)).fetchall()
        scored = []
        for memory_id, text, blob in rows:
            vector = np.frombuffer(blob, dtype=np.float32)
            score = float(np.dot(query_vec, vector) / (np.linalg.norm(query_vec) * np.linalg.norm(vector) + 1e-12))
            scored.append({'id': memory_id, 'text': text, 'score': score})
        return sorted(scored, key=lambda item: item['score'], reverse=True)[:k]

memory = LongTermMemory()
memory.upsert_preference('demo-user', 'travel_style', '喜欢安静、人少、步行友好的地方', '用户明确表达')
print(memory.preferences('demo-user'))
# await memory.remember('demo-user', '上次去苏州时更喜欢园林，不喜欢排队很久的网红店。')
# print(await memory.recall('demo-user', '周末去哪里比较合适？'))


### 核心实验


In [ ]:
class Document(BaseModel):
    id: str
    text: str
    source: str

class InMemoryVectorStore:
    def __init__(self) -> None:
        self.documents: list[Document] = []
        self.vectors: np.ndarray | None = None

    async def add(self, documents: list[Document]) -> None:
        vectors = await memory.embed([doc.text for doc in documents])
        array = np.asarray(vectors, dtype=np.float32)
        array /= np.linalg.norm(array, axis=1, keepdims=True) + 1e-12
        self.documents.extend(documents)
        self.vectors = array if self.vectors is None else np.vstack([self.vectors, array])

    async def search(self, query: str, k: int = 5, min_score: float = 0.25) -> list[dict[str, Any]]:
        if self.vectors is None:
            return []
        query_vec = np.asarray((await memory.embed([query]))[0], dtype=np.float32)
        query_vec /= np.linalg.norm(query_vec) + 1e-12
        scores = self.vectors @ query_vec
        indices = np.argsort(scores)[::-1][:k]
        return [
            {'id': self.documents[i].id, 'text': self.documents[i].text, 'source': self.documents[i].source, 'score': float(scores[i])}
            for i in indices if scores[i] >= min_score
        ]

class KnowledgeSearchArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    query: str = Field(min_length=2, max_length=300)
    k: int = Field(default=5, ge=1, le=8)

vector_store = InMemoryVectorStore()

async def search_knowledge_base(args: KnowledgeSearchArgs) -> list[dict[str, Any]]:
    return await vector_store.search(args.query, k=args.k)

rag_registry = ToolRegistry()
rag_registry.register(RegisteredTool(
    'search_knowledge_base',
    '检索私有知识库。遇到需要依据内部资料的问题时调用；结果不足时可换查询，最多三次。',
    KnowledgeSearchArgs,
    search_knowledge_base,
))
rag_agent = MinimalAgent(rag_registry, AgentLimits(max_steps=7))

# docs = [
#     Document(id='1', text='退款申请必须在购买后七天内提交。', source='policy.md#refund'),
#     Document(id='2', text='数字商品一经下载不支持无理由退款。', source='policy.md#digital'),
# ]
# await vector_store.add(docs)
# print(await rag_agent.run('下载过的数字商品还能申请七天无理由退款吗？请引用来源。'))


## 3. 观察与验证

核心代码中的真实 API 调用默认被注释。先运行无需额度的断言或定义单元格；确认输出和预期一致后，再逐行取消示例注释。


## 4. 代码讲解

示例先把检索做成可单独测试的组件，再接入 Agent。学习时先直接调用 `vector_store.search`，确认召回正确，再启用真实模型调用。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 加入 metadata 字段和过滤条件
- 记录每轮查询并阻止重复检索
- 构造一个证据不足时应拒答的问题

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 检索为空时不会编造内部事实
- [ ] 回答能标注 source
- [ ] 检索循环存在明确上限

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `08_多Agent协作.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。
